In [1]:
import numpy as np
from scipy.stats import norm

# Performance Metrics
mu = 60    # Mean response time (ms)
sigma = 5  # Standard deviation (ms)

# Ймовірність "просідання": Який відсоток часу гравець буде бачити менше 50 FPS? (Це критично для геймплею).
low_fps = 50
prob_low = norm.cdf(low_fps, mu, sigma)

# Ймовірність "ідеальної плавності": Який відсоток часу FPS буде знаходитись у діапазоні від 55 до 65?
prob_ideal = norm.cdf(65, mu, sigma) - norm.cdf(55, mu, sigma)

# Визначення аномалії: Якщо моніторинг зафіксував 78 FPS, чи є це статистичною аномалією (використовуй Z-score і правило $3\sigma$)?
detected = 78
pdf_detected = norm.pdf(detected, mu, sigma)

# Z-score
z_detected = (detected - mu) / sigma

print(f"Low FPS P(X<50): {prob_low:.4f}")
print(f"Ideal scanrio P(55<X<=65): {prob_ideal:.4f}")
print(f"Density in 78FPS: {pdf_detected:.4f}") # looks like anomaly cause 0.0001


if abs(z_detected) > 3:
    print(f"Z-score: {z_detected} => it's anomaly")
else:
    print(f"Z-score: {z_detected} => it's NOT anomaly")
    

Low FPS P(X<50): 0.0228
Ideal scanrio P(55<X<=65): 0.6827
Density in 78FPS: 0.0001
Z-score: 3.6 => it's anomaly


In [2]:
# скажи мені рівень FPS, нижче якого опускаються лише 5% найгірших випадків. Нам треба встановити 'мінімальні системні вимоги'"
worst_5 = norm.ppf(0.05, mu, sigma)
print(f"The worst 5% - {worst_5} FPS") # => MIN requirements are ~52 FPS

The worst 5% - 51.77573186524263 FPS


In [3]:
from scipy.stats import norm

mu = 170
sigma = 10

# 1. Точкова щільність (PDF) - рідко потрібна сама по собі
density_at_170 = norm.pdf(170, mu, sigma) 
print(f"Щільність у точці 170 (пік): {density_at_170:.4f}")

# 2. Ймовірність діапазону (CDF) - ТЕ, ЩО ТОБІ ТРЕБА
# Яка ймовірність, що студент має зріст від 160 до 180 см?
# (Це якраз правило 1 сигми, має бути ~0.68)
prob_range = norm.cdf(180, mu, sigma) - norm.cdf(160, mu, sigma)

print(f"Ймовірність зросту 160-180 см: {prob_range:.4f}")

Щільність у точці 170 (пік): 0.0399
Ймовірність зросту 160-180 см: 0.6827


Задача: Система моніторингу температури GPUУяви, що ти пишеш софт для майнінг-ферми або геймерського ПК.

Середня температура GPU під навантаженням: $\mu = 70°C$.

Стандартне відхилення: $\sigma = 5°C$.

Тобі потрібно розрахувати критичну межу температури.

Бизнес-правило таке: "Ми хочемо виділити 1% найбільш гарячих випадків, щоб надіслати сповіщення про перегрів".Тобто, тобі треба знайти таке число $X$, щоб ймовірність того, що температура вища за $X$, дорівнювала 0.01 (або 1%).

In [4]:
from scipy.stats import norm

mu = 70
sigma = 5

# 1. Знайди температуру, яка відсікає ВЕРХНІЙ 1% (використовуй ppf)
# Підказка: передай в ppf значення 0.99
critical_temp = norm.ppf(0.99, mu, sigma)# Твій код тут

# 2. Перевір себе: розрахуй ймовірність того, що температура вища за critical_temp
# Підказка: 1 - cdf(critical_temp)
check_prob = 1 - norm.cdf(critical_temp, mu, sigma) # Твій код тут

print(f"Критична температура (Top 1%): {critical_temp:.2f}°C")
print(f"Перевірка ймовірності: {check_prob:.4f}")

Критична температура (Top 1%): 81.63°C
Перевірка ймовірності: 0.0100


### LDA (example)

In [5]:
import pandas as pd
import numpy as np

# Твої зібрані дані (Ping у мілісекундах)
data = {
    'Ping': [115, 118, 120, 114, 119, 122, 125, 130, 128, 135, 132, 140, 129],
    'Type': ['Wired', 'Wired', 'Wired', 'Wired', 'Wired', 'Wired', 
             'WiFi', 'WiFi', 'WiFi', 'WiFi', 'WiFi', 'WiFi', 'WiFi']
}
df = pd.DataFrame(data)

# Нове значення для класифікації
x_new = 124

In [6]:
#  Classes
df_wired = df[df['Type'] == 'Wired']
df_wifi = df[df['Type'] == 'WiFi']

# Priors
p_wired = len(df_wired) / len(df)
print(f"Prior Probability for Wired: {p_wired}") # 0.4615

p_wifi = len(df_wifi) / len(df)
print(f"Prior probability for WiFi: {p_wifi}") # 0.5385

Prior Probability for Wired: 0.46153846153846156
Prior probability for WiFi: 0.5384615384615384


In [7]:
# Mean and Standard Deviation
# Wired
mu_wired = df_wired['Ping'].mean()
std_wired = df_wired['Ping'].std()
# WiFi
mu_wifi = df_wifi['Ping'].mean()
std_wifi = df_wifi['Ping'].std()

In [18]:
import math
# Likelihood
pdf_wired = norm.pdf(x_new, mu_wired, std_wired)
pdf_wifi = norm.pdf(x_new, mu_wifi, std_wifi)

def norm_distr(x, mu, sigma):
    # Здесь sigma — это стандартное отклонение. В квадрат оно возводится только внутри экспоненты.
    return (1 / (sigma * math.sqrt(2 * math.pi))) * math.exp(-0.5 * ((x - mu) / sigma) ** 2)

# print(f"Likelihood 'Wired' — {pdf_wired:.4f}")
# print(f"Likelihood 'WiFi' — {pdf_wifi:.4f}")

print(f"Likelihood 'Wired' — {norm_distr(x_new, mu_wired, std_wired**2):.4f}")
print(f"Likelihood 'WiFi' — {norm_distr(x_new, mu_wifi, std_wifi**2):.4f}")

Likelihood 'Wired' — 0.0351
Likelihood 'WiFi' — 0.0155


In [13]:
# Posterior
p_h_wired = (pdf_wired * p_wired)/(pdf_wired * p_wired + pdf_wifi * p_wifi)
print(f"Probability that {x_new} ms is Wider connection: {p_h_wired}")

p_h_wifi = (pdf_wifi * p_wifi)/(pdf_wired * p_wired + pdf_wifi * p_wifi)
print(f"Probability that {x_new} ms is WiFi connection: {p_h_wifi}")

Probability that 124 ms is Wider connection: 0.36834287449344943
Probability that 124 ms is WiFi connection: 0.6316571255065506


### Library implementtaion

In [15]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

ping = np.array(df['Ping'])
conn = np.array(df['Type'])

lda = LinearDiscriminantAnalysis()
lda.fit(ping.reshape(-1,1), conn)
x = np.array([x_new])

lda.predict_proba(x.reshape(-1,1))

array([[0.41786279, 0.58213721]])

In [14]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()
gnb.fit(ping.reshape(-1,1), conn)

# Перевіряємо твої 124 ms
print(gnb.predict_proba(x.reshape(-1,1)))

[[0.67623583 0.32376417]]


### Covariation Matrix

In [19]:
import numpy as np
import pandas as pd

# Наші дані: 5 розробників
# X (Години кодингу): 2, 4, 5, 7, 8
# Y (Кількість багів): 1, 3, 4, 6, 8
data = pd.DataFrame({
    'Hours': [2, 4, 5, 7, 8],
    'Bugs': [1, 3, 4, 6, 8]
})

X = data['Hours']
Y = data['Bugs']
n = len(data)

# --- КРОК 1: Ручний розрахунок (щоб зрозуміти математику) ---

# Середні значення
mu_x = X.mean() # 5.2
mu_y = Y.mean() # 4.4

# Дисперсії (Діагональ) - зверни увагу на ddof=1 (ділимо на n-1)
var_x = sum((X - mu_x)**2) / (n - 1)
var_y = sum((Y - mu_y)**2) / (n - 1)

# Коваріація (Побічна діагональ) - множимо відхилення
cov_xy = sum((X - mu_x) * (Y - mu_y)) / (n - 1)

print("--- Ручний розрахунок ---")
print(f"Var(X): {var_x:.2f}")
print(f"Var(Y): {var_y:.2f}")
print(f"Cov(X,Y): {cov_xy:.2f}")
print(f"Матриця:\n[[{var_x:.2f}, {cov_xy:.2f}]\n [{cov_xy:.2f}, {var_y:.2f}]]\n")

# --- КРОК 2: Production Way (через NumPy) ---

# np.cov приймає масив змінних. Ми передаємо X та Y.
# Важливо: NumPy за замовчуванням рахує вибірку (ділить на n-1)
cov_matrix = np.cov(X, Y)

print("--- NumPy np.cov ---")
print(cov_matrix)

--- Ручний розрахунок ---
Var(X): 5.70
Var(Y): 7.30
Cov(X,Y): 6.40
Матриця:
[[5.70, 6.40]
 [6.40, 7.30]]

--- NumPy np.cov ---
[[5.7 6.4]
 [6.4 7.3]]
